In [2]:
from keras.datasets import imdb
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.utils import pad_sequences
from keras import Sequential
from keras.layers import Dense,SimpleRNN,Embedding,Flatten

In [3]:
(X_train,y_train),(X_test,y_test) = imdb.load_data()

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
X_train = pad_sequences(X_train,padding='post',maxlen=50)
X_test = pad_sequences(X_test,padding='post',maxlen=50)

In [5]:
X_train.shape

(25000, 50)

In [7]:
model = Sequential()
model.add(Embedding(10000, 2,input_length=50))
model.add(SimpleRNN(32,return_sequences=False))
model.add(Dense(1, activation='sigmoid'))

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
history = model.fit(X_train, y_train,epochs=5,validation_data=(X_test,y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 10ms/step - acc: 0.5658 - loss: 0.6713 - val_acc: 0.7280 - val_loss: 0.5553
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - acc: 0.7885 - loss: 0.4574 - val_acc: 0.7929 - val_loss: 0.4466
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - acc: 0.8395 - loss: 0.3728 - val_acc: 0.7972 - val_loss: 0.4516
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 8ms/step - acc: 0.8587 - loss: 0.3406 - val_acc: 0.7883 - val_loss: 0.4665
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - acc: 0.8701 - loss: 0.3216 - val_acc: 0.7856 - val_loss: 0.5142


In [9]:
word_index = imdb.get_word_index()

def encode_review(text, num_words=10000, maxlen=50):
    words = text.lower().split()
    encoded = [1]  # 1 = start token (IMDB convention)
    for word in words:
        idx = word_index.get(word, 2) + 3  # +3 offset, 2 = unknown token
        encoded.append(idx if idx < num_words else 2)
    return pad_sequences([encoded], padding='post', maxlen=maxlen)

def predict_sentiment(text):
    encoded = encode_review(text)
    pred = model.predict(encoded, verbose=0)[0][0]
    sentiment = "Positive 🙂" if pred >= 0.5 else "Negative 🙁"
    print(f"Review: {text}")
    print(f"Score: {pred:.4f} -> {sentiment}\n")

predict_sentiment("This movie was absolutely fantastic, I loved every minute of it")
predict_sentiment("This was a terrible waste of time, the acting was awful")
predict_sentiment("The plot was boring and the characters were flat")


1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Review: This movie was absolutely fantastic, I loved every minute of it
Score: 0.5691 -> Positive 🙂

Review: This was a terrible waste of time, the acting was awful
Score: 0.0538 -> Negative 🙁

Review: The plot was boring and the characters were flat
Score: 0.0552 -> Negative 🙁



In [14]:
predict_sentiment("The visuals were stunning but the story was so predictable that it ruined the whole experience")

Review: The visuals were stunning but the story was so predictable that it ruined the whole experience
Score: 0.0805 -> Negative 🙁



In [15]:
predict_sentiment("Not the worst movie ever but definitely forgettable, nothing about it really stood out")
predict_sentiment("I expected to hate it based on the trailer but it actually surprised me with a solid ending")

Review: Not the worst movie ever but definitely forgettable, nothing about it really stood out
Score: 0.0550 -> Negative 🙁

Review: I expected to hate it based on the trailer but it actually surprised me with a solid ending
Score: 0.8607 -> Positive 🙂



In [16]:
predict_sentiment("I get why some people hate this . It's because of the political message and how some people think that you need get empathy for Arthur's madness. But come on that is not the point and it will never be. Enjoy this masterpiece because Joaquin Phoenix and Todd Phillips overdid themselves with this movie . The acting,music and cinematography are just amazing ! Please enjoy the movie without overthinking it.")

Review: I get why some people hate this . It's because of the political message and how some people think that you need get empathy for Arthur's madness. But come on that is not the point and it will never be. Enjoy this masterpiece because Joaquin Phoenix and Todd Phillips overdid themselves with this movie . The acting,music and cinematography are just amazing ! Please enjoy the movie without overthinking it.
Score: 0.9332 -> Positive 🙂



In [18]:
from google.colab import files

# Save both files
model.save('model.keras')

import json
word_index = imdb.get_word_index()
with open('word_index.json', 'w') as f:
    json.dump(word_index, f)

# Force-download them directly to your computer
files.download('model.keras')
files.download('word_index.json')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>